# Piping 

In [1]:
%load_ext dotenv
%dotenv

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

chat = ChatOpenAI(model="gpt-4",
                  seed=365,
                  max_completion_tokens=500,
                  temperature=0)

prompt_template = ChatPromptTemplate.from_template('''Tôi sắp nhận nôi một con {pet}, bạn hãy gợi ý cho con {pet} 3 cái tên thật hài hước.
''' + CommaSeparatedListOutputParser().get_format_instructions())

output_parser = CommaSeparatedListOutputParser()

pipe = prompt_template | chat | output_parser

pipe.invoke({"pet": "dog"})


['Bánh Mì', 'Cà Phê', 'Phở']

### Batch

In [16]:
prompt_template_for_batch = ChatPromptTemplate.from_messages(
    [("human", "Tôi sắp nhận nuôi một con {pet} thuộc giống loài {breed}. Hãy cho tôi một vài mẹo để dạy dỗ nó.")])

# prompt_template_for_batch

pipe_batch = prompt_template_for_batch | chat

# pipe_batch.invoke({"pet": "dog", "breed": "chó chăn cừu"})

pipe_batch.batch([
    {"pet": "chó", "breed": "chó chăn cừu"},
    {"pet": "mèo", "breed": "mèo rừng"}
])


[AIMessage(content='1. Bắt đầu sớm: Chó chăn cừu thông minh và học hỏi nhanh chóng, vì vậy bạn nên bắt đầu huấn luyện chúng từ khi còn nhỏ. \n\n2. Sử dụng lệnh đơn giản: Bắt đầu với các lệnh cơ bản như "ngồi", "đứng", "đợi", "đi" và "đến". Hãy đảm bảo rằng bạn luôn sử dụng cùng một từ ngữ cho mỗi lệnh để không làm rối trí chúng.\n\n3. Khen ngợi và thưởng: Khi chó của bạn thực hiện đúng lệnh, hãy khen ngợi và thưởng cho chúng. Điều này sẽ khích lệ chúng tiếp tục học hỏi và tuân thủ lệnh.\n\n4. Luyện tập thường xuyên: Chó chăn cừu cần nhiều hoạt động thể chất và trí óc để giữ cho chúng hạnh phúc và khỏe mạnh. Hãy đảm bảo rằng bạn dành thời gian hàng ngày để luyện tập và chơi với chúng.\n\n5. Kiên nhẫn: Mặc dù chó chăn cừu rất thông minh, nhưng việc huấn luyện vẫn cần thời gian và kiên nhẫn. Đừng nản lòng nếu chúng không ngay lập tức hiểu lệnh của bạn.\n\n6. Sử dụng dây dắt: Khi dạy chó chăn cừu, việc sử dụng dây dắt có thể giúp bạn kiểm soát', additional_kwargs={'refusal': None}, respons

### Stream

In [36]:
response = pipe_batch.stream(
    {"pet": "chó", "breed": "chó chăn cừu"}
)


In [37]:
for chunk in response:
    print (chunk.content, end="", flush=True)

1. Bắt đầu sớm: Chó chăn cừu thông minh và học hỏi nhanh chóng, vì vậy bạn nên bắt đầu huấn luyện chúng từ khi còn nhỏ. 

2. Sử dụng lệnh đơn giản: Bắt đầu với các lệnh cơ bản như "ngồi", "đứng", "đợi", "đi" và "đến". Hãy đảm bảo rằng bạn luôn sử dụng cùng một từ ngữ cho mỗi lệnh để không gây nhầm lẫn cho chó.

3. Khen ngợi và thưởng: Khi chó thực hiện đúng lệnh, hãy khen ngợi và thưởng cho chúng. Điều này giúp chó liên kết việc tuân thủ lệnh với những điều tích cực.

4. Luyện tập thường xuyên: Luyện tập mỗi ngày để chó có thể nhớ lâu các lệnh. Bạn không cần phải tập luyện trong thời gian dài, chỉ cần 15-20 phút mỗi ngày.

5. Kiên nhẫn: Đừng nóng giận hoặc trừng phạt chó nếu chúng không thực hiện đúng lệnh. Thay vào đó, hãy tiếp tục luyện tập và khích lệ chúng.

6. Sử dụng dây dắt: Khi dạy chó đi dạo, hãy sử dụng dây dắt để giữ chúng ở gần bạn và ngăn chúng chạy lung tung.

7. Tập dượt chúng với các tình huống khác nhau: Để chó chăn c

In [39]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chat_prompt_templates_tools = ChatPromptTemplate.from_template('''Liệt kê 5 tools quan trọng nhất của {job title}. 
Câu trả lời chỉ bao gồm danh sách các tools.
''')

# chat_prompt_templates_tools

str_output_parser = StrOutputParser()


In [40]:
tools_chain = chat_prompt_templates_tools | chat | str_output_parser
tools_chain.invoke({'job title': 'Data science'})

'1. Python\n2. R\n3. SQL\n4. Tableau\n5. Hadoop'

In [41]:
chat_prompt_templates_strategies = ChatPromptTemplate.from_template('''Với mỗi tool hãy cho tôi chiến lược để học nó một cách hiệu quả. 
Đây là danh sách các {tools}''')

strategies_chain = chat_prompt_templates_strategies | chat | str_output_parser

learning_tools_strategies = tools_chain | {'tools': RunnablePassthrough()} | strategies_chain

learning_tools_strategies.invoke({'job title': 'Data science'})

'1. Python: \n   - Bắt đầu với việc hiểu cú pháp cơ bản của Python và cách sử dụng các loại dữ liệu khác nhau.\n   - Học về vòng lặp, điều kiện và hàm trong Python.\n   - Tiếp theo, hãy tìm hiểu về OOP (Lập trình hướng đối tượng) trong Python.\n   - Học cách làm việc với thư viện như Numpy, Pandas, Matplotlib, Scikit-learn để phân tích dữ liệu và xây dựng mô hình học máy.\n   - Thực hành bằng cách giải quyết các vấn đề thực tế và tham gia các dự án nhỏ.\n\n2. R: \n   - Bắt đầu với cú pháp cơ bản của R và cách sử dụng các loại dữ liệu khác nhau.\n   - Học về vòng lặp, điều kiện và hàm trong R.\n   - Học cách làm việc với các gói như dplyr, ggplot2, caret để phân tích dữ liệu và xây dựng mô hình học máy.\n   - Thực hành bằng cách giải quyết các vấn đề thực tế và tham gia các dự án nhỏ.\n\n3. SQL: \n   - Bắt đầu với cú pháp cơ bản của SQL, cách tạo, truy vấn và cập nhật cơ sở dữ liệu.\n   - Học về các loại join, subquery, và các hàm tổng hợp.\n   - Thực hành bằng cách giải quyết các vấn đ